# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [1]:
%uv pip install torch triton

Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 25ms
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch

import triton
import triton.language as tl
from triton.runtime import driver

DEVICE = triton.runtime.driver.active.get_active_torch_device()

def is_hip():
  return triton.runtime.driver.active.get_current_target().backend=="hip"

def is_cdna():
  return is_hip() and triton.runtime.driver.active.get_current_target().arch in ('gfx940', 'gfx941', 'gfx942',
                                                                                  'gfx90a', 'gfx908')
  
# simple (numerically stabilized) softmax op
def naive_softmax(x): # x E R ^ (MxN)
  """row-wise softmax of X using pytorch
  we need to subtract the max element to avoid overflow"""

  # read MN elemnts ; write M elements
  x_max = x.max(dim=1)[0]
  # read MN+M elements ; write MN elements
  z = x - x_max[:, None]
  # read MN elements ; write MN elements
  numerator = torch.exp(z)
  # read MN elements ; write M elements
  denominator = numerator.sum(dim=1)
  # read MN+M elements ; write MN elements
  ret = numerator / denominator[:, None]
  # in total read 5MN+2M elements ; wrote 3MN+2M elements
  return ret

In [8]:
@triton.jit
def softmax_kernel(output_ptr, input_ptr, 
                  input_row_stride, output_row_stride,
                  n_rows, n_cols,
                   BLOCK_SIZE:tl.constexpr, num_stages: tl.constexpr):
    row_start = tl.program_id(0)
    row_step = tl.num_programs(0)
    for row_idx in tl.range(row_start, n_rows, row_step, num_stages=num_stages):
        # the stride represents how much needed to increase the pointer to advance 1 row
        row_start_ptr = input_ptr + row_idx * input_row_stride
        # 
        col_offsets = tl.arange(0, BLOCK_SIZE)
        input_ptrs = row_start_ptr + col_offsets
        # load the row, using mask since block size may be > than n cols
        mask = col_offsets < n_cols
        row = tl.load(input_ptrs, mask=mask, other=-float('inf'))
        # subtract max for avoid overflow
        row_minus_max = row - tl.max(row, axis=0)
    
        numerator = tl.exp(row_minus_max)
        denominator = tl.sum(numerator, axis=0)
        softmax_output = numerator / denominator
    
        # writin it back to memory
        output_row_start_ptr = output_ptr + row_idx * output_row_stride
        output_ptrs = output_row_start_ptr + col_offsets
        tl.store(output_ptrs, softmax_output, mask=mask)

Helper function that enqueues the kernel and its   arguments for any given input tensor

In [9]:
properties = driver.active.utils.get_device_properties(DEVICE.index)
NUM_SM = properties["multiprocessor_count"]
NUM_REGS = properties["max_num_regs"]
SIZE_SMEM = properties["max_shared_mem"]
WARP_SIZE = properties["warpSize"]
target = triton.runtime.driver.active.get_current_target()
kernels = {}

def softmax(x):
    n_rows, n_cols = x.shape
    # The block size of each loop iteration is the smallest power of two greater than the number of columns in `x`
    BLOCK_SIZE = triton.next_power_of_2(n_cols)

    num_warps = 8

    # # software pipelining stages
    num_stages = 4 if SIZE_SMEM > 20000 else 2

    # allocate output
    y = torch.empty_like(x)

    # pre compiling the kernel to get register usage and compute thread occupancy
    kernel = softmax_kernel.warmup(y, x, 
                                   x.stride(0), y.stride(0), 
                                   n_rows, n_cols,
                                   BLOCK_SIZE = BLOCK_SIZE, num_stages=num_stages, 
                                   num_warps=num_warps, grid=(1, ))
    kernel._init_handles()
    n_regs = kernel.n_regs
    size_smem = kernel.metadata.shared
    if is_hip(): # (?)
        NUM_GPRS = NUM_REGS
        if is_cdna():
            NUM_GPRS = NUM_REGS * 2

        MAX_NUM_THREADS = properties["max_threads_per_sm"]
        max_num_waves = MAX_NUM_THREADS // WARP_SIZE
        occupancy = min(NUM_GPRS // WARP_SIZE // n_regs, max_num_waves) // num_warps
    else:
        occupancy = NUM_REGS // (n_regs * WARP_SIZE * num_warps)
    occupancy = min(occupancy, SIZE_SMEM // size_smem)
    num_programs = NUM_SM * occupancy

    num_programs = min(num_programs, n_rows)

    # Create a number of persistent programs.
    kernel[(num_programs, 1, 1)](y, x, x.stride(0), y.stride(0), n_rows, n_cols, BLOCK_SIZE, num_stages)
    return y

    

### Unit Test

In [10]:
# torch.manual_seed(0)
# x = torch.randn(1823, 781, device=DEVICE)
# y_triton = softmax(x)
# y_torch = torch.softmax(x, axis=1)
# assert torch.allclose(y_triton, y_torch), (y_triton, y_torch)

### Benchmark

In [11]:
# @triton.testing.perf_report(
#     triton.testing.Benchmark(
#         x_names=['N'],  # argument names to use as an x-axis for the plot
#         x_vals=[128 * i for i in range(2, 100)],  # different possible values for `x_name`
#         line_arg='provider',  # argument name whose value corresponds to a different line in the plot
#         line_vals=['triton', 'torch', 'naive_softmax'],  # possible values for `line_arg``
#         line_names=["Triton", "Torch", "Naive Softmax"],  # label name for the lines
#         styles=[('blue', '-'), ('green', '-'), ('red', '-')],  # line styles
#         ylabel="GB/s",  # label name for the y-axis
#         plot_name="softmax-performance",  # name for the plot. Used also as a file name for saving the plot.
#         args={'M': 4096},  # values for function arguments not in `x_names` and `y_name`
#     ))
# def benchmark(M, N, provider):
#     x = torch.randn(M, N, device=DEVICE, dtype=torch.float32)
#     stream = getattr(torch, DEVICE.type).Stream()
#     getattr(torch, DEVICE.type).set_stream(stream)
#     if provider == 'torch':
#         ms = triton.testing.do_bench(lambda: torch.softmax(x, axis=-1))
#     if provider == 'triton':
#         ms = triton.testing.do_bench(lambda: softmax(x))
#     if provider == 'naive_softmax':
#         ms = triton.testing.do_bench(lambda: naive_softmax(x))
#     gbps = lambda ms: 2 * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
#     return gbps(ms)


# benchmark.run(show_plots=True, print_data=True)

OutOfResources: out of resource: shared memory, Required: 196640, Hardware limit: 101376. Reducing block sizes or `num_stages` may help.

### Compare every row sums to 1 & against PyTorch

In [12]:
torch.manual_seed(0)

test_shapes = [
    (3, 5),
    (17, 127),
    (31, 781),
    (8, 1025),
]

for M, N in test_shapes:
    x = torch.rand((M, N), device=DEVICE, dtype=torch.float32)
    y = softmax(x)
    row_sums = y.sum(dim=1)
    max_sum_error = (row_sums - 1.0).abs().max().item()

    print(
        f"shape={M}x{N:<4} "
        f"block={triton.next_power_of_2(N):<4} "
        f"max_row_sum_error={max_sum_error:.3e}"
    )

    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5, rtol=1e-5)



shape=3x5    block=8    max_row_sum_error=1.192e-07
shape=17x127  block=128  max_row_sum_error=1.192e-07
shape=31x781  block=1024 max_row_sum_error=1.192e-07
shape=8x1025 block=2048 max_row_sum_error=1.192e-07


In [13]:
torch.manual_seed(1)

test_shapes = [
    (3, 5),
    (17, 127),
    (31, 781),
    (8, 1025),
    (4, 4097),
]

for M, N in test_shapes:
    x = torch.randn((M, N), device=DEVICE, dtype=torch.float32)

    y_triton = softmax(x)
    y_torch = torch.softmax(x, dim=1)

    absolute_error = (y_triton - y_torch).abs()
    max_abs_error = absolute_error.max().item()
    mean_abs_error = absolute_error.mean().item()

    torch.testing.assert_close(
        y_triton,
        y_torch,
        atol=1e-6,
        rtol=1e-4,
    )

    print(
        f"shape={M}x{N:<4} "
        f"max error={max_abs_error:.3e}, "
        f"mean error={mean_abs_error:.3e}"
    )

shape=3x5    max error=7.451e-09, mean error=1.024e-09
shape=17x127  max error=7.451e-09, mean error=4.651e-10
shape=31x781  max error=5.588e-09, mean error=1.028e-10
shape=8x1025 max error=7.451e-09, mean error=8.304e-11
shape=4x4097 max error=4.657e-10, mean error=1.675e-11


### Backward Softmax

In [16]:
torch.manual_seed(2)

x = torch.randn(
    (32, 781),
    device=DEVICE,
    dtype=torch.float32,
    requires_grad=True
)

g = torch.randn_like(x)

# Pytroch autograd result
y = torch.softmax(x, dim=1)
y.backward(g)
dx_autograd = x.grad

# Manual formula
with torch.no_grad():
    dot = (g*y).sum(dim=1, keepdim=True)
    dx_formula = y * (g-dot)

torch.testing.assert_close(dx_formula, dx_autograd, atol=1e-6, rtol=1e-4)

print("Maximum backward error:", (dx_formula-dx_autograd).abs().max().item())
print("Maximum gradient row sum:", dx_formula.sum(dim=1).abs().max().item())

Maximum backward error: 3.725290298461914e-09
Maximum gradient row sum: 2.9802322387695312e-08


In [24]:
@triton.jit
def softmax_backward_kernel(
    dx_ptr, y_ptr, g_ptr,
    dx_row_stride, y_row_stride, g_row_stride,
    n_cols, BLOCK_SIZE:tl.constexpr
):
    row_idx = tl.program_id(0)
    col_offs = tl.arange(0, BLOCK_SIZE)
    mask = col_offs < n_cols

    y_ptrs = y_ptr + row_idx * y_row_stride + col_offs
    g_ptrs = g_ptr + row_idx * g_row_stride + col_offs

    y = tl.load(y_ptrs, mask=mask, other=0.0)
    g = tl.load(g_ptrs, mask=mask, other=0.0)

    dot = tl.sum(y*g, axis=0)
    dx = y*(g-dot)

    dx_ptrs = dx_ptr + row_idx * dx_row_stride + col_offs
    tl.store(dx_ptrs, dx, mask=mask)

wrapper

In [25]:
def softmax_backward(y, g):
    assert y.ndim==2
    assert g.shape==y.shape
    assert y.is_contiguous()
    assert g.is_contiguous()

    n_rows, n_cols = y.shape
    block_size = triton.next_power_of_2(n_cols)

    num_warps = 4
    if block_size >= 2048:
        num_warps = 8

    dx = torch.empty_like(y)

    softmax_backward_kernel[(n_rows,)] (
        dx, y, g,
        dx.stride(0), y.stride(0), g.stride(0),
        n_cols, BLOCK_SIZE=block_size, num_warps=num_warps
    )

    return dx

In [27]:
torch.manual_seed(3)

x = torch.randn(
    (1823, 781),
    device=DEVICE,
    dtype=torch.float32,
    requires_grad=True
)

g = torch.randn_like(x)

# Reference
y_torch = torch.softmax(x, dim=1)
y_torch.backward(g)
dx_ref = x.grad.detach()

# triton forward + backward
with torch.no_grad():
    y_triton = softmax(x.detach())
    dx_triton = softmax_backward(y_triton, g)

torch.testing.assert_close(
    dx_triton, dx_ref,
    atol=2e-6, rtol=2e-4
)

print("Maximum error:",
      (dx_triton - dx_ref).abs().max().item())

print("Maximum gradient row sum:",
      dx_triton.sum(dim=1).abs().max().item())

Maximum error: 1.4901161193847656e-08
Maximum gradient row sum: 3.5390257835388184e-08


now completed the full softmax computation:  
Forward:  
x -> max -> subtract -> exp -> sum -> divide -> y  
Backward:  
(y, dy) -> sum(y x dy) -> y x (dy - sum) -> dx